In [1]:
# AUTOMATIC SPEECH RECOGNITION - TEXT AND SPEECH ANALYSIS APP
# Run this entire code in ONE Google Colab cell.

import subprocess
import sys

print("Installing required libraries...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "faster-whisper", "gradio", "transformers", "gTTS"
])

import re
import tempfile
import collections
import gradio as gr
import torch
from faster_whisper import WhisperModel
from transformers import pipeline
from gtts import gTTS

print("Loading speech recognition model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

asr_model = WhisperModel(
    "base", device=device, compute_type=compute_type
)

print("Loading sentiment analysis model...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if device == "cuda" else -1
)

stopwords = set("""
a an the is am are was were be been being to of in on at for
and or but with by from it this that i you he she we they have
has had do does did not my your our their
""".split())


def analyze_audio(audio_path, create_speech):
    if not audio_path:
        return "Please upload an audio file.", "", "", None, None

    try:
        # Speech recognition
        segments, info = asr_model.transcribe(audio_path)
        transcript = " ".join(
            segment.text.strip() for segment in segments
        ).strip()

        if not transcript:
            return "No speech detected. Try another audio file.", "", "", None, None

        # Text analysis
        words = re.findall(r"\b[\w']+\b", transcript.lower())
        sentences = [
            s for s in re.split(r"(?<=[.!?])\s+", transcript.strip())
            if s
        ]

        word_count = len(words)
        character_count = len(transcript)
        sentence_count = len(sentences)

        # Keyword extraction
        counts = collections.Counter(
            w for w in words if w not in stopwords and len(w) > 2
        )
        keywords = ", ".join(
            f"{word} ({count})"
            for word, count in counts.most_common(10)
        ) or "No keywords found"

        # Sentiment analysis
        sentiment = sentiment_model(transcript[:1500])[0]
        sentiment_text = (
            f"{sentiment['label']} "
            f"({sentiment['score'] * 100:.1f}% model confidence)"
        )

        report = f"""
AUTOMATIC SPEECH RECOGNITION REPORT

Detected language: {info.language}
Audio duration: {info.duration:.2f} seconds

TEXT ANALYSIS
Word count: {word_count}
Character count: {character_count}
Sentence count: {sentence_count}

SENTIMENT ANALYSIS
Result: {sentiment_text}

KEYWORD ANALYSIS
{keywords}
"""

        # Save transcript for download
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".txt",
            delete=False, encoding="utf-8"
        ) as f:
            f.write(transcript)
            text_file = f.name

        # Optional text-to-speech
        speech_file = None
        if create_speech:
            speech_file = tempfile.NamedTemporaryFile(
                suffix=".mp3", delete=False
            ).name
            gTTS(text=transcript, lang="en").save(speech_file)

        return transcript, report, keywords, text_file, speech_file

    except Exception as error:
        return f"Processing error: {error}", "", "", None, None


# Application interface
with gr.Blocks(title="Speech and Text Analysis System") as app:
    gr.Markdown("""
    # 🎙️ Automatic Speech Recognition System
    ### Text and Speech Analysis Application

    Upload an audio file to convert speech into text, analyze the text,
    identify sentiment, extract keywords, and optionally generate speech.
    """)

    audio = gr.Audio(
        sources=["upload", "microphone"],
        type="filepath",
        label="Upload Audio or Record Speech"
    )

    create_speech = gr.Checkbox(
        label="Convert recognized text back into English speech",
        value=False
    )

    button = gr.Button("Transcribe and Analyze", variant="primary")

    transcript_box = gr.Textbox(
        label="Recognized Text", lines=6
    )
    report_box = gr.Textbox(
        label="Analysis Report", lines=12
    )
    keywords_box = gr.Textbox(
        label="Extracted Keywords", lines=3
    )
    text_download = gr.File(label="Download Text File")
    speech_download = gr.File(label="Download Generated Speech")

    button.click(
        fn=analyze_audio,
        inputs=[audio, create_speech],
        outputs=[
            transcript_box,
            report_box,
            keywords_box,
            text_download,
            speech_download
        ]
    )

print("Application is ready. Wait for the interface link.")
app.launch(share=True)

Installing required libraries...
Loading speech recognition model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading sentiment analysis model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Application is ready. Wait for the interface link.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e1dcb9505b84c8d877.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
